# Installs

In [ ]:
# !python -m pip install plotly
# !python -m pip install pandas
# !python -m pip install numpy
# !python -m pip install --upgrade kaleido


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Modules and Functions

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from tqdm import tqdm

def create_geo_df(cities, cities_df):
    """Create a DF with 'original Name', 'standard name', 'longitude', 'latitude'
    :inputs 
        - cities: A list of city names
        - cities_df: a pandas df names alternate names and coordinates
    """
    lons = []
    lats = []
    cities_output = []
    cities_orig_name = []
    for city in tqdm(cities):
        if city in list(cities_df["Name"]):
            lat,lon = cities_df[cities_df["Name"]==city]["Coordinates"].iloc[0].split(",")
            lons.append(lon)
            lats.append(lat)
            cities_output.append(city)
            cities_orig_name.append(city)
        
        else:# Check for a cityname in the alternate names
            
            for city_names in cities_df["Alternate Names"]:
                if type(city_names) == str:  # there is one entry that is a float, somehow...
                    if city in city_names.split(","):
                        if city == "Tirano":
                            #  The correspondence mentions the Italian village, but Tirano finds a spelling variant of an Albanian city...
                            break
                        # save the old name also
                        cities_orig_name.append(city)
                        city_data = cities_df[cities_df["Alternate Names"]==city_names]
                        lat, lon = city_data["Coordinates"].iloc[0].split(",")
                        city = city_data["Name"].iloc[0]  # Use the now official name!!
                        lons.append(lon)
                        lats.append(lat)
                        cities_output.append(city)
                        # Break the loop, because I found
                        break
    
    return pd.DataFrame.from_dict({"city_orig_name":cities_orig_name, "city": cities_output, "lon": lons, "lat": lats})


def create_arrow_data(origin_point, coordinate_df):
    """Create a df for the arrows
    Inputs:
        - origin_point: a Tuple or list with two coordinates (lat, lon)
        - coordinate_df: and pandas df with columns lat and lon"""

    start_lats = []
    start_lons = []
    end_lats = []
    end_lons = []


    for coordinate in coordinate_df.iloc:
        start_lats.append(origin_point[1])
        start_lons.append(origin_point[0])
        end_lats.append(coordinate.lat)
        end_lons.append(coordinate.lon)
    return pd.DataFrame.from_dict({"start_lon": start_lons, "start_lat": start_lats,
                                                    "end_lon": end_lons, "end_lat": end_lats})

pd.set_option("display.max_colwidth", None)


## Data

In [3]:
# data from cities from europe:
df_europe = pd.read_csv("geonames-all-cities-with-a-population-1000.csv", sep=";")

# data that I need
df_europe = df_europe[df_europe["Population"]>10000]

# There's a Baden in Austria, but we most likely don't want that one...
df_europe = df_europe.drop(df_europe[(df_europe["Name"] == "Baden") & (df_europe["Country Code"] == "AT")].index)

# This city Aiud can also be called strassburg, but we don't want that:
df_europe.loc[df_europe["Name"] == "Aiud", "Alternate Names"] = df_europe.loc[df_europe["Name"] == "Aiud", "Alternate Names"].str.replace(r'Strassburg|Straßburg', '', regex=True)


In [4]:
# cities that Nägeli had corresponance with:
df_corres_cities = pd.read_csv("cities.csv", sep=";")
corres_cities = list(set(list(df_corres_cities['Ort'])))  # unique list

In [ ]:
# Creating the data that we need, matching the cities to the coordinates
naegeli_cities_df = create_geo_df(corres_cities, df_europe)

  0%|          | 0/216 [00:00<?, ?it/s]

100%|██████████| 216/216 [00:04<00:00, 47.74it/s]


In [10]:
naegeli_cities_df.head()

,city_orig_name,city,lon,lat
0,Hechingen,Hechingen,8.96317,48.35149
1,Bamberg,Bamberg,10.90067,49.89873
2,Orléans,Orléans,1.90389,47.90289
3,Konstanz,Konstanz,9.17582,47.66033
4,Altstätten,Altstätten,9.54746,47.37766


## Drawing the lines
If you draw lines, do it before the dots, otherwise the hover function won't work.

In [6]:
# Startet immer von Zürich
zh_coor = (8.55, 47.36667)
arrow_df = create_arrow_data(zh_coor, naegeli_cities_df)

fig = go.Figure()

lons = []
lats = []
lons = np.empty(3 * len(arrow_df))
lons[::3] = arrow_df['start_lon']
lons[1::3] = arrow_df['end_lon']
lons[2::3] = None
lats = np.empty(3 * len(arrow_df))
lats[::3] = arrow_df['start_lat']
lats[1::3] = arrow_df['end_lat']
lats[2::3] = None

fig.add_trace(
    go.Scattergeo(
        lon = lons,
        lat = lats,
        mode = 'lines',
        hoverinfo = "skip",
        line = dict(width = 0.5,color = 'red'),
        opacity = 0.5,
    )
)

fig.show()

## Drawing Dots

In [ ]:

# fig = go.Figure()  # Uncomment, if no Arrow data generated

# Add the cities
fig.add_trace(go.Scattergeo(
    lon = naegeli_cities_df['lon'],
    lat = naegeli_cities_df['lat'],
    hoverinfo = 'text',
    text = naegeli_cities_df['city'],
    mode = 'markers',  # show text: "markers+text"
    marker = dict(
        size = 2,
        color = 'rgb(255, 0, 0)',
        # bgcolor = 'rgb(255, 255, 255)',
        line = dict(
            width = 1,
            color = 'rgba(255, 0, 0, 0)'

        )
    )))

fig.show()

## Customize the Map

In [14]:
# einkreisen auf Europa: 
fig.update_layout(

    showlegend = False,
    geo = go.layout.Geo(
        scope = 'europe',
        projection_type = 'azimuthal equal area',
        showland = True,
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',# 'rgb(204, 204, 204)',
        
    ),
    height=700,
)

fig.show()

In [9]:
# fig.write_html("Naegeli_map.html")
fig.write_image("naegeli_map.svg", format="svg")
fig.write_image("naegeli_map.pdf", format="pdf")
